## 1. Importation des bibliothèques

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

## 2. Charger le dataset fusionné

In [2]:
data = pd.read_csv("./data/processed/merged_dataset.csv")

## 3. Vérifier les dimensions

In [3]:
print(data.shape)

(2830743, 79)


## 4.1 Vérification des valeurs dupliquées

In [4]:
duplicates = data.duplicated().sum()

print(f"Nombre de lignes dupliquées : {duplicates:,}")

Nombre de lignes dupliquées : 308,381


## 4.2 Vérification des valeurs manquantes

In [5]:
missing_values = pd.DataFrame({
    "Missing Values": data.isnull().sum(),
    "Percentage (%)": (data.isnull().sum()/len(data)*100).round(2)
})

missing_values[missing_values["Missing Values"] > 0]

,Missing Values,Percentage (%)
Flow Bytes/s,1358,0.05


## 4.3 Vérification des valeurs infinies

In [6]:
import numpy as np

infinite_values = np.isinf(data.select_dtypes(include=np.number)).sum()

infinite_values[infinite_values > 0]

Flow Bytes/s       1509
 Flow Packets/s    2867
dtype: int64

## 4.4) Nettoyage des colonnes constantes

In [7]:
constant_cols = [col for col in data.columns if data[col].nunique() == 1]
print(f"Colonnes constantes supprimées ({len(constant_cols)}) : {constant_cols}")

data = data.drop(columns=constant_cols)
print(f"Nouvelle dimension : {data.shape}")

Colonnes constantes supprimées (8) : [' Bwd PSH Flags', ' Bwd URG Flags', 'Fwd Avg Bytes/Bulk', ' Fwd Avg Packets/Bulk', ' Fwd Avg Bulk Rate', ' Bwd Avg Bytes/Bulk', ' Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Nouvelle dimension : (2830743, 71)


## 4.5) Traitement des valeurs infinies et manquantes

In [8]:
num_cols = data.select_dtypes(include=[np.number]).columns

# Remplacement des infinis par NaN
data[num_cols] = data[num_cols].replace([np.inf, -np.inf], np.nan)

print("Valeurs manquantes avant imputation :")
print(data.isnull().sum()[data.isnull().sum() > 0])

# Imputation par la médiane (robuste aux outliers)
for col in ['Flow Bytes/s', 'Flow Packets/s']:
    if col in data.columns:
        data[col] = data[col].fillna(data[col].median())

print("\nValeurs manquantes après imputation :", data.isnull().sum().sum())

Valeurs manquantes avant imputation :
Flow Bytes/s       2867
 Flow Packets/s    2867
dtype: int64

Valeurs manquantes après imputation : 2867


## 4.6) Correction des valeurs négatives incohérentes

In [9]:
cols_a_verifier = ['Flow Duration', 'Fwd Header Length', 'Bwd Header Length',
                    'Fwd Header Length.1', 'min_seg_size_forward']

for col in cols_a_verifier:
    if col in data.columns:
        n_neg = (data[col] < 0).sum()
        print(f"{col} : {n_neg} valeurs négatives")

# Suppression des lignes avec des valeurs négatives incohérentes
for col in cols_a_verifier:
    if col in data.columns:
        data = data[data[col] >= 0]

print(f"\nDimension après suppression des valeurs négatives : {data.shape}")


Dimension après suppression des valeurs négatives : (2830743, 71)


## 4.7) Suppression des doublons

In [10]:
print(f"Dimension avant suppression : {data.shape}")

data = data.drop_duplicates()

print(f"Dimension après suppression des doublons : {data.shape}")

Dimension avant suppression : (2830743, 71)


Dimension après suppression des doublons : (2522362, 71)


## 4.8) Réduction de la redondance (corrélation)

In [11]:
# Recalcul de la matrice de corrélation sur les données nettoyées
num_cols = data.select_dtypes(include=[np.number]).columns
corr = data[num_cols].corr()

seuil = 0.9
strong_corr = corr.abs().unstack().sort_values(ascending=False)
strong_corr = strong_corr[(strong_corr < 1) & (strong_corr > seuil)]
strong_corr = strong_corr.drop_duplicates()

print(f"Nombre de paires fortement corrélées (>0.9) : {len(strong_corr)}")

# Identification des colonnes à supprimer (on garde une variable par paire redondante)
cols_a_supprimer = set()
for (col1, col2) in strong_corr.index:
    if col1 not in cols_a_supprimer:
        cols_a_supprimer.add(col2)

print(f"Colonnes redondantes supprimées ({len(cols_a_supprimer)}) : {cols_a_supprimer}")

data = data.drop(columns=list(cols_a_supprimer))
print(f"\nNouvelle dimension : {data.shape}")

Nombre de paires fortement corrélées (>0.9) : 59
Colonnes redondantes supprimées (23) : {'Fwd IAT Total', 'Bwd Packet Length Max', ' Bwd Packet Length Std', ' Idle Max', ' Active Min', ' Flow IAT Mean', ' Subflow Bwd Bytes', ' Flow IAT Max', ' Packet Length Std', ' Fwd Packet Length Max', ' Subflow Fwd Bytes', ' Packet Length Mean', 'Idle Mean', ' Average Packet Size', ' Flow Packets/s', ' Idle Min', ' Avg Bwd Segment Size', ' ECE Flag Count', 'Subflow Fwd Packets', ' Fwd IAT Max', ' Total Length of Bwd Packets', ' Bwd IAT Mean', ' Max Packet Length'}

Nouvelle dimension : (2522362, 48)


## 4.9) Encodage de la variable cible (Label)

In [12]:
import sys
!{sys.executable} -m pip install scikit-learn

from sklearn.preprocessing import LabelEncoder


[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python3.11 -m pip install --upgrade pip


In [15]:

# Nettoyage des caractères mal encodés
data[' Label'] = data[' Label'].str.replace('\ufffd', '-', regex=False)

le = LabelEncoder()
data['Label_encoded'] = le.fit_transform(data[' Label'])

# Table de correspondance
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Correspondance des classes :")
for label, code in label_mapping.items():
    print(f"  {code} -> {label}")

Correspondance des classes :
  0 -> BENIGN
  1 -> Bot
  2 -> DDoS
  3 -> DoS GoldenEye
  4 -> DoS Hulk
  5 -> DoS Slowhttptest
  6 -> DoS slowloris
  7 -> FTP-Patator
  8 -> Heartbleed
  9 -> Infiltration
  10 -> PortScan
  11 -> SSH-Patator
  12 -> Web Attack - Brute Force
  13 -> Web Attack - Sql Injection
  14 -> Web Attack - XSS


## sauvegarde de la data nettoyée

In [16]:
import os
data.to_csv("./data/processed/cleaned_dataset.csv", index=False)
print("Dataset nettoyé sauvegardé.")

Dataset nettoyé sauvegardé.
